# Upload the data


---

upload the dataset from Roboflow


In [1]:
import os
HOME = os.getcwd()


In [2]:
! pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [3]:
import ultralytics
ultralytics.checks()


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="bgZz4XsPKrewZX7Qhbab")
project = rf.workspace("ahmeds-workspace-slwao").project("clamp-bfcse")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to clamp-1 in yolov11:: 100%|██████████| 65/65 [00:00<00:00, 2859.13it/s]


# The Model

In [10]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLO11m model
model = YOLO("yolo11m.pt")

# Train the model
results = model.train(data="clamp-1/data.yaml", epochs=10, imgsz=640  )


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=clamp-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

# Predict

In [6]:
import cv2
from collections import Counter

In [13]:


# Input and output video paths
input_video = "Task.mp4"
output_video = "predict12.mp4"

# Open input video
cap = cv2.VideoCapture(input_video)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame  , conf =.15 , iou=0 )

    annotated_frame = results[0].plot()
    class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

    # Count labels
    counts = Counter(class_ids)



    y = 80

    for class_id, count in counts.items():

        label_name = model.names[class_id]

        text = f"{label_name}: {count}"

        cv2.putText(
            annotated_frame,
            text,
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 0, 0),
            2
        )

    y += 35
    out.write(annotated_frame)


    # Press q to quit early
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break


cap.release()
out.release()
# cv2.destroyAllWindows()


0: 384x640 20 clamps, 23.9ms
Speed: 3.5ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 clamps, 23.9ms
Speed: 3.3ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 clamps, 23.9ms
Speed: 3.5ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 clamps, 24.2ms
Speed: 3.0ms preprocess, 24.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 clamps, 23.9ms
Speed: 3.3ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 clamps, 23.9ms
Speed: 3.4ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 clamps, 23.9ms
Speed: 3.0ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 clamps, 23.9ms
Speed: 3.4ms preprocess, 23.9ms inference, 1.3ms postprocess per image at shape (